# Memory Profiling: ¿dónde se está gastando la RAM?

En el vídeo anterior aprendimos a medir el **tiempo**. Ahora toca el otro gran recurso
que un programa consume y que mucha gente nunca llega a mirar: la **memoria**.

La pregunta que queremos responder es:

> **¿Qué objetos están consumiendo memoria?**

Igual que con el tiempo, la regla es **medir antes de optimizar**. Un programa puede ir
rápido pero "comerse" toda la RAM (o al revés), y a menudo el culpable no es el que
imaginamos. Veremos, de lo más simple a lo más completo:

- **`sys.getsizeof`**: el tamaño de un objeto... y sus límites.
- El problema del **tamaño profundo** (*deep size*) y la herramienta `pympler`.
- **`memory_profiler`**: consumo de memoria **línea a línea**.
- **`tracemalloc`**: rastreo de asignaciones incluido en Python (fugas, comparaciones).
- Un ejercicio revelador: **generadores vs listas**.

> 📚 Documentación de referencia:
> **[`sys.getsizeof`](https://docs.python.org/es/3/library/sys.html#sys.getsizeof)** y
> **[`tracemalloc`](https://docs.python.org/es/3/library/tracemalloc.html)**.

---

# Parte 1: El tamaño de un objeto con `sys.getsizeof`

## 1. ¿Cuánto ocupa esto?

La forma más directa de preguntar "¿cuánta memoria ocupa este objeto?" es
`sys.getsizeof`, que devuelve el tamaño **en bytes**:

In [1]:
import sys

print("int:  ", sys.getsizeof(42), "bytes")
print("str:  ", sys.getsizeof("hola"), "bytes")
print("lista vacía:", sys.getsizeof([]), "bytes")
print("lista 3 elem:", sys.getsizeof([1, 2, 3]), "bytes")

int:   28 bytes
str:   45 bytes
lista vacía: 56 bytes
lista 3 elem: 88 bytes


Hasta aquí parece sencillo. Pero `sys.getsizeof` tiene una **limitación importante**:
mide el tamaño del objeto "en sí", pero **no cuenta la memoria de los objetos a los que
hace referencia** por dentro. Y eso, con las estructuras anidadas, engaña mucho.

---

# Parte 2: El tamaño profundo (*deep size*)

## 2. Por qué `getsizeof` engaña

Una lista no guarda los elementos "dentro" de sí misma, sino **referencias** (punteros)
a ellos. `sys.getsizeof` mide esos punteros, no lo que apuntan. Mira lo que pasa con
una lista de listas:

In [2]:
import sys

fila = list(range(1000))                       # 1000 enteros
matriz = [list(range(1000)) for _ in range(1000)]  # 1000 filas = 1.000.000 de enteros

print("Una fila (1000 enteros):     ", sys.getsizeof(fila), "bytes")
print("La matriz (1.000.000 enteros):", sys.getsizeof(matriz), "bytes")

Una fila (1000 enteros):      8056 bytes
La matriz (1.000.000 enteros): 8856 bytes


¡La "matriz" con un millón de enteros dice ocupar casi lo mismo que una sola fila! Es
absurdo, claro: `getsizeof(matriz)` solo cuenta los **1000 punteros** a las filas, no
las 1000 filas en sí. La memoria real es unas mil veces mayor.

Para conocer el **tamaño profundo** (el objeto *más* todo lo que cuelga de él)
necesitamos una herramienta que recorra las referencias. La más conocida es
**`pympler`**, con su función `asizeof`. Se instala como dependencia de desarrollo:

```bash
uv add --dev pympler
```

In [3]:
from pympler import asizeof

print(asizeof.asizeof(fila))    # ~ 40 KB  (la fila y sus 1000 enteros)
print(asizeof.asizeof(matriz))  # ~ 32 MB  (¡ahora sí cuenta las 1000 filas y su contenido!)

40056
31849080



> 💡 Regla práctica: usa `sys.getsizeof` para una idea rápida de objetos simples, pero
> cuando tengas **estructuras anidadas** (listas de listas, diccionarios de objetos...)
> necesitas el *tamaño profundo* (`pympler.asizeof`) para no llevarte una falsa
> impresión.

---

# Parte 3: `memory_profiler`, memoria línea a línea

## 3. ¿En qué línea crece la memoria?

Igual que `cProfile` nos decía en qué **función** se iba el tiempo, **`memory_profiler`**
nos dice en qué **línea** crece la memoria. Es de las herramientas que más sorprenden,
porque señala con el dedo la línea exacta que dispara el consumo.

Se instala con:

```bash
uv add --dev memory-profiler
```

Se decora con `@profile` la función que queremos analizar (en un archivo, p. ej.
`programa.py`):

In [ ]:
from memory_profiler import profile

@profile
def cargar_datos():
    a = [0] * 1_000_000          # reservamos una lista grande
    b = [0] * 10_000_000         # y otra aún mayor
    del b                        # liberamos la segunda
    return a

cargar_datos()

Y se ejecuta indicando el módulo `memory_profiler`:

```bash
python -m memory_profiler programa.py
```

La salida muestra una tabla **línea a línea**:

```
Line #    Mem usage    Increment   Line Contents
================================================
     2     38.1 MiB     38.1 MiB   @profile
     3     38.1 MiB      0.0 MiB   def cargar_datos():
     4     45.7 MiB      7.6 MiB       a = [0] * 1_000_000
     5    121.991 MiB    76.3 MiB       b = [0] * 10_000_000
     6     45.7 MiB    -76.3 MiB       del b
     7     45.7 MiB      0.0 MiB       return a
```

- **Mem usage**: memoria total del proceso en esa línea.
- **Increment**: cuánto **subió o bajó** la memoria en esa línea concreta.

De un vistazo se ve que la línea 5 es la que dispara el consumo (+76 MiB) y que el
`del b` lo recupera. Esto es **extremadamente útil** para localizar la línea culpable.

---

# Parte 4: `tracemalloc`, rastreo incluido en Python

## 4. Top de asignaciones de memoria

`tracemalloc` viene **de serie** con Python (no hay que instalar nada) y permite
rastrear todas las asignaciones de memoria. Se activa con `tracemalloc.start()`, y en
cualquier momento podemos tomar una **"foto"** (*snapshot*) del estado de la memoria y
pedir las asignaciones más grandes:

In [5]:
import tracemalloc

tracemalloc.start()

# código que asigna memoria
muchos_textos = [f"elemento {i}" for i in range(100_000)]
numeros = list(range(100_000))

snapshot = tracemalloc.take_snapshot()
top = snapshot.statistics("lineno")  # agrupado por línea de código

print("Top 3 líneas que más memoria han asignado:")
for stat in top[:3]:
    print("  ", stat)

tracemalloc.stop()

Top 3 líneas que más memoria han asignado:
   /tmp/ipykernel_594039/3813468707.py:6: size=6142 KiB, count=100001, average=63 B
   /tmp/ipykernel_594039/3813468707.py:7: size=3898 KiB, count=99745, average=40 B
   /home/alvaro/.local/share/virtualenvs/pontia-dkEw0-DC/lib/python3.12/site-packages/IPython/core/history.py:1053: size=918 B, count=4, average=230 B


Cada línea del resultado indica el **archivo:línea** y cuánta memoria se asignó allí.
Así localizamos el origen real del consumo.

## 5. Comparar *snapshots*: detectar fugas

La función estrella de `tracemalloc` es **comparar dos *snapshots***: uno antes y otro
después de un trozo de código. La diferencia nos dice qué memoria ha **crecido**, que
es justo la pista para encontrar una **fuga** (memoria que se acumula y no se libera).

In [6]:
import tracemalloc

tracemalloc.start()
antes = tracemalloc.take_snapshot()

# Simulamos un proceso que va acumulando datos (una posible fuga)
cache = []
for i in range(50_000):
    cache.append("dato " + str(i))

despues = tracemalloc.take_snapshot()

# ¿Qué ha crecido entre 'antes' y 'despues'?
diferencias = despues.compare_to(antes, "lineno")

print("Mayores crecimientos de memoria:")
for stat in diferencias[:3]:
    print("  ", stat)

tracemalloc.stop()

Mayores crecimientos de memoria:
   /tmp/ipykernel_594039/107613630.py:9: size=2913 KiB (+2913 KiB), count=50001 (+50001), average=60 B
   /home/alvaro/.local/share/virtualenvs/pontia-dkEw0-DC/lib/python3.12/site-packages/IPython/core/history.py:1053: size=1008 B (+1008 B), count=3 (+3), average=336 B
   /home/alvaro/.pyenv/versions/3.12.10/lib/python3.12/tracemalloc.py:560: size=304 B (+304 B), count=2 (+2), average=152 B


> 💡 Comparar *snapshots* es la técnica clásica para cazar **fugas de memoria**: tomas
> una foto, ejecutas la parte sospechosa (por ejemplo, procesar muchas peticiones), y
> al comparar ves qué estructuras siguen creciendo sin parar.

---

# Parte 5: Generadores vs listas (el ejercicio revelador)

## 6. La misma tarea, muchísima menos memoria

Vamos a crear "un millón de números" de dos formas y medir la memoria real con `tracemalloc`.

Primero, una **lista**: construye y guarda **todos** los números a la vez en memoria.

In [10]:
import tracemalloc

tracemalloc.start()
lista = [x for x in range(1_000_000)]  # crea y almacena 1.000.000 de enteros
actual, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Lista: {actual / 1024 / 1024:.1f} MB en memoria")
print("Primeros elementos:", lista[:5])
print("  suma =", sum(lista))
del lista

Lista: 38.6 MB en memoria
Primeros elementos: [0, 1, 2, 3, 4]
  suma = 499999500000


Ahora, un **generador**: no guarda los números, sino la **receta** para producirlos uno
a uno según se vayan pidiendo. Fíjate en los paréntesis `(...)` en lugar de los
corchetes `[...]`.

In [11]:
import tracemalloc

tracemalloc.start()
generador = (x for x in range(1_000_000))  # NO crea los números todavía
actual, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Generador: {actual} bytes en memoria")
print("Podemos recorrerlo igual, pero sin guardarlo todo de golpe:")
print("  suma =", sum(generador))  # produce los números sobre la marcha

Generador: 1726 bytes en memoria
Podemos recorrerlo igual, pero sin guardarlo todo de golpe:
  suma = 499999500000


La diferencia es brutal: la lista ocupa **decenas de MB**, mientras que el generador
ocupa apenas **unos cientos de bytes**, porque no almacena nada: genera cada valor
cuando se le pide y lo olvida.

> 💡 Si solo necesitas **recorrer** los datos una vez (sumar, filtrar, procesar uno a
> uno), un **generador** te ahorra una cantidad enorme de memoria. Usa una **lista**
> cuando de verdad necesites tenerlos todos a la vez (acceder por índice, recorrerlos
> varias veces, etc.).

---

## 7. Resumen

En este vídeo hemos aprendido a responder "¿dónde se está gastando la RAM?":

1. **`sys.getsizeof`** da el tamaño de un objeto, pero **no cuenta sus referencias
   internas**: engaña con estructuras anidadas.
2. Para el **tamaño profundo** (el objeto y todo lo que cuelga de él) se usa
   `pympler.asizeof`.
3. **`memory_profiler`** muestra el consumo **línea a línea** (`@profile` +
   `python -m memory_profiler`): ideal para señalar la línea culpable.
4. **`tracemalloc`** (incluido en Python) rastrea asignaciones, da el **top** de
   consumo y permite **comparar *snapshots*** para cazar fugas.
5. **Generadores vs listas**: si solo vas a recorrer los datos, un generador ahorra
   muchísima memoria frente a una lista.

| Herramienta | Para qué | ¿Incluida en Python? |
|-------------|----------|----------------------|
| `sys.getsizeof` | Tamaño "superficial" de un objeto | Sí |
| `pympler.asizeof` | Tamaño profundo (con referencias internas) | No (`uv add --dev pympler`) |
| `memory_profiler` | Consumo de memoria línea a línea | No (`uv add --dev memory-profiler`) |
| `tracemalloc` | Top de asignaciones y comparación de snapshots | Sí |